# 04 — Claim Severity: Baseline and Gamma GLM

**Phase 6**, step 1 of 4: build the claim-level modelling table, fit a trivial baseline, and fit the first Gamma GLM.

Severity asks a different question from frequency: not "how often does a policy claim" but "given a claim happened, how much did it cost." That conditioning is not just conceptual -
it changes what the training data even looks like. A policy with zero claims has **no severity observation at all** (not a severity of zero), so this notebook works from the claim-level severity table (26,444 claims), not the 678,013-row policy table.

Phase 3's severity EDA already set expectations here: no rating factor showed a convincing univariate effect on severity, and the distribution is extremely skewed (top 1% of claims hold 38% of total value). This phase checks whether a proper multivariate model finds more than the univariate view did - but going in, a strong result is not expected.

In [1]:
import numpy as np
import pandas as pd

from auto_pricing.severity import (
    SEVERITY_FORMULA,
    baseline_severity,
    build_severity_table,
    fit_gamma_glm,
    predict_severity,
)

sev = pd.read_parquet("../data/processed/severity_clean.parquet")
model_table = pd.read_parquet("../data/processed/model_table.parquet")

severity_table = build_severity_table(sev, model_table)
print(f"severity table: {severity_table.shape}")
print(severity_table["split"].value_counts())

severity table: (26444, 20)
split
train         18523
test           3974
validation     3947
Name: count, dtype: int64


**18,523 / 3,947 / 3,974** claims across train/validation/test - matching *exactly* the claim counts `scripts/split_data.py` printed back in Phase 4 step 1. That match confirms the split-inheritance mechanism worked correctly: each claim was assigned to its **policy's** split (via the `IDpol` join), never split independently at the claim level - the same leakage-prevention rule from Phase 4, applied one level down.

In [2]:
train = severity_table[severity_table["split"] == "train"].copy()
print(f"train claims: {len(train):,}")

train claims: 18,523


## 1. Baseline severity

In [3]:
baseline = baseline_severity(train)
print(f"baseline (mean) severity: \u20ac{baseline:,.2f}")

baseline (mean) severity: €2,375.68


The simplest possible severity model: predict this one number for every claim, regardless of who or what caused it. Any real model needs to beat this - but Phase 3 already suggests it might not beat it by much.

## 2. Gamma GLM

In [4]:
print(SEVERITY_FORMULA)

ClaimAmount ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + AreaOrdinal + LogDensity + VehGasBinary + C(RegionGrouped) + C(VehBrandGrouped) + VehPower


In [5]:
result = fit_gamma_glm(train)
print(f"converged: {result.converged}")
print(f"parameters: {len(result.params)}")
print(f"deviance: {result.deviance:.1f}")
print(f"dispersion (scale) estimate: {result.scale:.2f}")

converged: True
parameters: 48
deviance: 29265.2
dispersion (scale) estimate: 21.92


A **Gamma** distribution with a **log link** is the standard choice for a continuous, positive, right-skewed outcome like claim cost - the same logic as Poisson for counts, matched to a different shape of data. No exposure offset here: unlike claim frequency, a claim's cost has no mechanical relationship to how long the policy was observed before the claim happened.

The large dispersion (scale) estimate (~22) reflects exactly the severe skew found in Phase 3 (skewness ~110) - a Gamma distribution's dispersion parameter directly controls how heavy its tail is, and this data has a genuinely heavy one.

### Does the full model actually beat the baseline?

In [6]:
null_result = fit_gamma_glm(train, formula="ClaimAmount ~ 1")  # intercept-only = the baseline, refit as a GLM for a fair deviance comparison

deviance_reduction = (null_result.deviance - result.deviance) / null_result.deviance
print(f"null (baseline-only) deviance: {null_result.deviance:.1f}")
print(f"full model deviance:           {result.deviance:.1f}")
print(f"deviance reduction: {deviance_reduction:.1%}")

null (baseline-only) deviance: 34346.0
full model deviance:           29265.2
deviance reduction: 14.8%


**A 14.8% deviance reduction** - real, but modest, exactly matching Phase 3's expectation that severity would be much harder to explain from policy characteristics than frequency was. For comparison, Phase 5's frequency model beat its own baseline far more decisively. This is not a failure of the modelling approach - it is consistent with the plan's own expectation (and scikit-learn's official insurance tutorial, the project's methodological reference point) that severity is intrinsically harder to predict from these variables than frequency is.

### Sanity check: predicted vs. observed total (training data)

In [7]:
pred_train = predict_severity(result, train)
print(f"predicted total claim cost: \u20ac{pred_train.sum():,.2f}")
print(f"observed total claim cost:  \u20ac{train['ClaimAmount'].sum():,.2f}")
print(f"ratio: {pred_train.sum() / train['ClaimAmount'].sum():.4f}")

predicted total claim cost: €42,719,139.91
observed total claim cost:  €44,004,810.78
ratio: 0.9708


Off by about 2.9% - and this is expected, not a bug, for the same reason Phase 5's Negative Binomial model didn't exactly match its training total either. Only a Poisson GLM with a canonical link has the mathematical guarantee of an exact match; Gamma's estimating equations don't share that specific property. A small, consistent gap like this is normal; the frequency model's earlier 2x+ discrepancy (before that bug was fixed) was not.

## 3. Summary of step 1

- Claim-level table built (26,444 claims), correctly inheriting each claim's policy's split   assignment - verified against Phase 4's own split counts, not just assumed correct.
- Baseline severity: \u20ac2,375.68 (training mean).
- Gamma GLM fit successfully; the full feature set improves on the baseline by **14.8%   deviance reduction** - a real but modest improvement, consistent with Phase 3's finding   that severity has little relationship to the available rating factors.
- Predicted and observed training totals are close (ratio 0.971), consistent with a properly   fit model of this type.

### Next
Step 2: a lognormal alternative model, and the large-loss sensitivity analysis Phase 3's cleaning policy explicitly deferred to this phase (does the model's story change if the largest few claims are excluded?).